In [3]:
install.packages("e1071")
install.packages("caret")
install.packages("dplyr")
install.packages("naivebayes")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘shape’, ‘future.apply’, ‘numDeriv’, ‘progressr’, ‘SQUAREM’, ‘diagram’, ‘lava’, ‘prodlim’, ‘iterators’, ‘clock’, ‘gower’, ‘hardhat’, ‘ipred’, ‘sparsevctrs’, ‘timeDate’, ‘foreach’, ‘ModelMetrics’, ‘plyr’, ‘pROC’, ‘recipes’, ‘reshape2’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [6]:
# ====================================================
# Comparar Naive Bayes con y sin Validación Cruzada
# ====================================================

# Cargar librerías necesarias
library(e1071)      # Para naiveBayes
library(caret)      # Para RMSE, validación cruzada y otros
library(dplyr)      # Para manipulación de datos

# Cargar datos preprocesados
train_data <- read.csv("train_preprocessed.csv", stringsAsFactors = TRUE)
test_data  <- read.csv("test_preprocessed.csv", stringsAsFactors = TRUE)

factor_vars <- names(train_data)[sapply(train_data, is.factor)]
for (var in factor_vars) {
  if (var %in% names(test_data)) {
    test_data[[var]] <- factor(test_data[[var]], levels = levels(train_data[[var]]))
  }
}

# Eliminar filas con NA en ambos conjuntos
train_data <- train_data[complete.cases(train_data), ]
test_data  <- test_data[complete.cases(test_data), ]

# Definir los predictores: todas las variables excepto "SalePrice"
predictors <- setdiff(names(train_data), "SalePrice")

# Definir los bins para la variable "SalePrice" (discretización)
n_bins <- 50
unique_vals <- length(unique(train_data$SalePrice))
n_bins <- min(n_bins, unique_vals - 1)
bins <- quantile(train_data$SalePrice, probs = seq(0, 1, length.out = n_bins + 1), na.rm = TRUE)
bins <- unique(bins)  # Evitar cortes repetidos
train_data$SalesPrice_bin <- cut(train_data$SalePrice, breaks = bins, include.lowest = TRUE, dig.lab = 10)
bin_centers <- (head(bins, -1) + tail(bins, -1)) / 2

# ====================================================
# Versión sin Validación Cruzada
# ====================================================

# Entrenar el modelo Naive Bayes sin validación cruzada
nb_model_no_cv <- naiveBayes(SalesPrice_bin ~ ., data = train_data[, c(predictors, "SalesPrice_bin")])

# Predecir en el conjunto de prueba sin validación cruzada
nb_pred_probs_no_cv <- predict(nb_model_no_cv, newdata = test_data[, predictors], type = "raw")

# Calcular las predicciones como valor esperado
nb_pred_no_cv <- apply(nb_pred_probs_no_cv, 1, function(prob_vec) sum(prob_vec * bin_centers))

# Calcular el RMSE para el modelo sin validación cruzada
rmse_no_cv <- RMSE(nb_pred_no_cv, test_data$SalePrice)
cat("RMSE sin validación cruzada:", rmse_no_cv, "\n")

# ====================================================
# Versión con Validación Cruzada
# ====================================================

control_cv <- trainControl(method = "cv", number = 10)

# Entrenar el modelo Naive Bayes con validación cruzada
grid <- expand.grid(laplace = c(0, 1, 2),
                    usekernel = c(TRUE, FALSE),
                    adjust = c(0.5, 1, 2))

results_cv <- data.frame(laplace = numeric(),
                         usekernel = logical(),
                         adjust = numeric(),
                         RMSE = numeric())

for (i in 1:nrow(grid)) {
  params <- grid[i, ]
  cat("Evaluando con validación cruzada: laplace =", params$laplace,
      " usekernel =", params$usekernel,
      " adjust =", params$adjust, "\n")

  nb_model_cv <- train(SalesPrice_bin ~ .,
                       data = train_data[, c(predictors, "SalesPrice_bin")],
                       method = "naive_bayes",
                       trControl = control_cv,
                       tuneGrid = params)

  # Predecir en el conjunto de prueba con el modelo de validación cruzada
  nb_pred_probs_cv <- predict(nb_model_cv, newdata = test_data[, predictors], type = "prob")

  nb_pred_cv <- apply(nb_pred_probs_cv, 1, function(prob_vec) sum(prob_vec * bin_centers))

  # Calcular el RMSE para esta combinación
  rmse_cv <- RMSE(nb_pred_cv, test_data$SalePrice)
  cat("RMSE con validación cruzada:", rmse_cv, "\n")

  # Almacenar el resultado
  results_cv <- rbind(results_cv, cbind(params, RMSE = rmse_cv))
}

# Mostrar los resultados ordenados por RMSE de la validación cruzada
results_cv <- results_cv[order(results_cv$RMSE), ]
cat("Resultados de la búsqueda de hiperparámetros con validación cruzada:\n")
print(results_cv)

# ====================================================
# Comparación de Resultados
# ====================================================

cat("\nComparación de RMSE:\n")
cat("RMSE sin validación cruzada:", rmse_no_cv, "\n")
cat("Mejor RMSE con validación cruzada:", min(results_cv$RMSE), "\n")


RMSE sin validación cruzada: 0.6245234 
Evaluando con validación cruzada: laplace = 0  usekernel = TRUE  adjust = 0.5 
RMSE con validación cruzada: 0.6231303 
Evaluando con validación cruzada: laplace = 1  usekernel = TRUE  adjust = 0.5 
RMSE con validación cruzada: 0.6231303 
Evaluando con validación cruzada: laplace = 2  usekernel = TRUE  adjust = 0.5 
RMSE con validación cruzada: 0.6231303 
Evaluando con validación cruzada: laplace = 0  usekernel = FALSE  adjust = 0.5 
RMSE con validación cruzada: 0.7472468 
Evaluando con validación cruzada: laplace = 1  usekernel = FALSE  adjust = 0.5 
RMSE con validación cruzada: 0.7472468 
Evaluando con validación cruzada: laplace = 2  usekernel = FALSE  adjust = 0.5 
RMSE con validación cruzada: 0.7472468 
Evaluando con validación cruzada: laplace = 0  usekernel = TRUE  adjust = 1 
RMSE con validación cruzada: 0.6305801 
Evaluando con validación cruzada: laplace = 1  usekernel = TRUE  adjust = 1 
RMSE con validación cruzada: 0.6305801 
Evaluando